# Coordinated data augmentation

Apply one spatial transformation to aligned modalities and a segmentation.
This synthetic example requires no downloads. ANTsTorch's
`augment_image_modalities` delegates augmentation to ANTsPy on CPU; the image
dataset uses the same helper when preparing training samples.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

In [ ]:
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
x, y = np.meshgrid(np.linspace(-1, 1, 48), np.linspace(-1, 1, 48), indexing="ij")
phantom = np.exp(-6*((x-0.15)**2 + (y+0.1)**2)).astype("float32")
labels = ((x-0.15)**2 + (y+0.1)**2 < 0.25).astype("float32")
reference = ants.from_numpy(phantom, spacing=(1., 1.))
segmentation = reference.new_image_like(labels)
# Duplicate modalities make shared spatial sampling easy to check.
modalities = [reference.clone(), reference.clone()]
augmented, augmented_labels = antstorch.augment_image_modalities(
    modalities, reference_image=reference, transform_type="affine",
    sd_affine=0.03, sd_deformation=0,
    noise_model=None, sd_simulated_bias_field=0, sd_histogram_warping=0,
    segmentation_image=[segmentation],
)
assert len(augmented) == 2 and augmented_labels is not None
np.testing.assert_allclose(augmented[0].numpy(), augmented[1].numpy(), atol=1e-6)
assert np.isin(augmented_labels.numpy(), [0, 1]).all()
for image in [*augmented, augmented_labels]:
    assert image.shape == reference.shape and np.isfinite(image.numpy()).all()
    np.testing.assert_allclose(image.spacing, reference.spacing)
    np.testing.assert_allclose(image.origin, reference.origin)
    np.testing.assert_allclose(image.direction, reference.direction)

## Inspect the transformed label boundary
The segmentation uses label interpolation, while intensities use image
interpolation. Intensity normalization still occurs inside augmentation even
with noise, bias and histogram perturbations disabled. Seeds help reproduce
Python/NumPy choices; native ANTs randomness need not be identical across runs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
for ax, image, mask, title in zip(
    axes, [reference, *augmented], [segmentation, augmented_labels, augmented_labels],
    ["Original", "Augmented modality A", "Augmented modality B"],
):
    ax.imshow(image.numpy().T, origin="lower", cmap="gray")
    ax.contour(mask.numpy().T, levels=[0.5], colors="red", linewidths=1)
    ax.set_title(title); ax.axis("off")
plt.show()

## Add intensity variation
Spatial augmentation should stay synchronized across modalities and labels.
Intensity perturbations can differ between channels; they must not modify
the label identities. Here noise and histogram warping are enabled without
adding a spatial deformation.

In [ ]:
perturbed, _ = antstorch.augment_image_modalities(
    modalities, reference_image=reference, transform_type="affine", sd_affine=0,
    noise_model="additivegaussian", noise_parameters=(0., 0.03),
    sd_simulated_bias_field=0, sd_histogram_warping=0.02,
)
assert all(np.isfinite(image.numpy()).all() for image in perturbed)
fig, axes = plt.subplots(1, 2, figsize=(7, 3), constrained_layout=True)
for ax, image, title in zip(axes, perturbed, ["Intensity variant A", "Intensity variant B"]):
    ax.imshow(image.numpy().T, origin="lower", cmap="gray")
    ax.set_title(title); ax.axis("off")
plt.show()

For a dataset, replace the duplicate images with genuinely aligned modalities
and pass their matching segmentation as a one-item list. Choose perturbation
ranges that match plausible acquisition differences rather than maximizing
visual variation.